# Pruebas CRUD — TPO Airbnb (Ingeniería de Datos II)

Este notebook ejercita **todos los requerimientos funcionales** del enunciado
contra la API REST del backend (`http://localhost:3000/api`). Cada celda
ejecuta llamadas reales: cualquier cambio se ve inmediatamente en el frontend
(`http://localhost:5173`) tras un refresh.

**Antes de correr**, asegurate de que el stack esté arriba:

```bash
docker compose up -d
```

Las **variables editables** están al inicio de cada celda — modificalas si el
profesor pide cambiar algún valor puntual.

---

## Índice

1. Gestión de usuarios
2. Gestión de propiedades
3. Gestión de reservas
4. Gestión de pagos
5. Gestión de reseñas


## 0. Setup

Importamos `requests`, definimos el endpoint base y helpers para imprimir
respuestas en formato legible.

In [1]:
import requests
import json
from datetime import date, timedelta

# ── Variables editables ─────────────────────────────────────────────────
API = "http://localhost:3000/api"   # cambiar si el backend está en otro puerto/host

# IDs de seed (vienen pre-cargados con docker compose up)
ADMIN_ID   = "00000000-0000-0000-0000-000000000001"
HOST1_ID   = "11111111-0001-0001-0001-000000000001"   # Ana Gómez
HOST2_ID   = "11111111-0002-0001-0001-000000000002"   # Carlos Pérez
GUEST1_ID  = "22222222-0001-0001-0001-000000000001"   # Lucía
GUEST2_ID  = "22222222-0002-0001-0001-000000000002"   # Martín
PROP1_ID   = "33333333-0001-0001-0001-000000000001"   # Depto Palermo
PROP2_ID   = "33333333-0002-0001-0001-000000000002"   # Casa Córdoba
PROP3_ID   = "33333333-0003-0001-0001-000000000003"   # Loft Rosario

DEFAULT_PASSWORD = "demo1234"   # password de todos los usuarios seed

# ── Helpers ─────────────────────────────────────────────────────────────
def show(resp, max_items=5):
    """Imprime el status + el body de la respuesta de forma legible."""
    print(f"HTTP {resp.status_code} · {resp.request.method} {resp.request.url}")
    try:
        body = resp.json()
        # Si la respuesta es una lista, truncar para no inundar la salida
        if isinstance(body.get("data"), list) and len(body["data"]) > max_items:
            n = len(body["data"])
            body = {**body, "data": body["data"][:max_items], "_truncated": f"{n - max_items} items más"}
        print(json.dumps(body, indent=2, ensure_ascii=False, default=str))
    except Exception:
        print(resp.text[:500])
    return resp

# Health check rápido — debe responder OK antes de seguir
show(requests.get(f"{API}/health"))

HTTP 200 · GET http://localhost:3000/api/health
{
  "ok": true,
  "data": {
    "status": "running",
    "mode": "multi-db",
    "databases": [
      "mongodb",
      "postgres",
      "cassandra",
      "neo4j"
    ]
  }
}


<Response [200]>

---

# 1. Gestión de Usuarios

Endpoints involucrados:
- `POST /api/auth/register` — registrar usuario (huésped o anfitrión) con bcrypt
- `GET  /api/usuarios/:id` — consultar perfil (incluye propiedades si es anfitrión)
- `PUT  /api/usuarios/:id` — actualizar perfil
- `GET  /api/usuarios?tipo=anfitrion` — listar por tipo


## 1.1 Registrar usuario como huésped o anfitrión

> *El sistema debe permitir registrar usuarios como huésped o anfitrión.*

Crea un usuario nuevo. El backend hashea el password con bcrypt antes de
guardarlo y sincroniza el nodo en Neo4j.

In [2]:
# ── Variables editables ─────────────────────────────────────────────────
nuevo_usuario = {
    "nombre":   "Juan Pérez Test",
    "email":    "juan.test@mail.com",
    "password": "secreto123",
    "tipo":     "huesped",            # "huesped" o "anfitrion"
    "telefono": "11-9999-8888",
    "bio":      "Usuario creado desde el notebook"
}

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.post(f"{API}/auth/register", json=nuevo_usuario))

# Guardamos el ID para usarlo en celdas siguientes (si el email ya existía, falla con 409)
if r.ok:
    NUEVO_USER_ID = r.json()["data"]["id"]
    print(f"\n✅ ID generado: {NUEVO_USER_ID}")
else:
    print("\n⚠️  Si el email ya existe, modificá 'email' arriba y volvé a correr la celda.")

HTTP 201 · POST http://localhost:3000/api/auth/register
{
  "ok": true,
  "data": {
    "id": "cdb1a229-38d3-442e-86f1-cd696b0b62f5",
    "nombre": "Juan Pérez Test",
    "email": "juan.test@mail.com",
    "telefono": "11-9999-8888",
    "tipo": "huesped",
    "bio": "Usuario creado desde el notebook",
    "promedio_rating": 0,
    "created_at": "2026-06-06T16:47:51.926Z"
  }
}

✅ ID generado: cdb1a229-38d3-442e-86f1-cd696b0b62f5


## 1.2 Consultar el perfil de un usuario

> *El sistema debe permitir consultar y actualizar el perfil de un usuario.*

Devuelve los datos públicos del usuario (sin `password_hash`). Si el usuario es
anfitrión, también incluye el array `propiedades` con todas las que publicó.

In [3]:
# ── Variables editables ─────────────────────────────────────────────────
user_id = "11111111-0001-0001-0001-000000000001"   # cambiar a otro UUID para consultar otro usuario

# ── Llamada ─────────────────────────────────────────────────────────────
show(requests.get(f"{API}/usuarios/{user_id}"))

HTTP 200 · GET http://localhost:3000/api/usuarios/11111111-0001-0001-0001-000000000001
{
  "ok": true,
  "data": {
    "id": "11111111-0001-0001-0001-000000000001",
    "bio": "Host con foco en estadías urbanas.",
    "created_at": "2025-01-01T00:00:00.000Z",
    "email": "ana.host@mail.com",
    "nombre": "Ana Gómez",
    "promedio_rating": 5,
    "telefono": "1111-2222",
    "tipo": "anfitrion",
    "propiedades": [
      {
        "id": "33333333-0001-0001-0001-000000000001",
        "anfitrion_id": "11111111-0001-0001-0001-000000000001",
        "cantidad_huespedes": 3,
        "descripcion": "Cerca de bares, subte y zona gastronómica.",
        "estado": "activa",
        "imagen": "https://images.unsplash.com/photo-1522708323590-d24dbb6b0267?auto=format&fit=crop&w=900&q=80",
        "precio_noche": 60000,
        "promedio_rating": 5,
        "servicios": [
          "wifi",
          "aire acondicionado",
          "cocina"
        ],
        "tipo": "departamento",
        "tit

<Response [200]>

## 1.3 Actualizar el perfil de un usuario

> *El sistema debe permitir consultar y actualizar el perfil de un usuario.*

El backend acepta cualquier subset de campos. No se puede cambiar `id` ni
`tipo` (están protegidos en el endpoint).

In [4]:
# ── Variables editables ─────────────────────────────────────────────────
user_id = HOST1_ID
patch = {
    "telefono": "11-5555-1111",
    "bio":      "Bio actualizada desde el notebook",
}

# ── Llamada ─────────────────────────────────────────────────────────────
show(requests.put(f"{API}/usuarios/{user_id}", json=patch))

HTTP 200 · PUT http://localhost:3000/api/usuarios/11111111-0001-0001-0001-000000000001
{
  "ok": true,
  "data": {
    "id": "11111111-0001-0001-0001-000000000001",
    "bio": "Bio actualizada desde el notebook",
    "created_at": "2025-01-01T00:00:00.000Z",
    "email": "ana.host@mail.com",
    "nombre": "Ana Gómez",
    "promedio_rating": 5,
    "telefono": "11-5555-1111",
    "tipo": "anfitrion"
  }
}


<Response [200]>

## 1.4 Ver el perfil de un anfitrión con sus propiedades

> *El sistema debe permitir visualizar el perfil de un anfitrión junto con las
> propiedades que gestiona.*

El mismo `GET /api/usuarios/:id` cuando el usuario es anfitrión incluye el
array completo de sus propiedades activas.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
anfitrion_id = HOST1_ID   # Ana Gómez

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.get(f"{API}/usuarios/{anfitrion_id}"), max_items=20)

if r.ok:
    data = r.json()["data"]
    props = data.get("propiedades", [])
    print(f"\n📊 {data['nombre']} tiene {len(props)} propiedades")

---

# 2. Gestión de Propiedades

Endpoints:
- `POST   /api/propiedades` — publicar
- `GET    /api/propiedades?ciudad=...&tipo=...&precioMax=...&anfitrion_id=...&lat=...&lng=...&radioKm=...`
- `GET    /api/propiedades/:id` — detalle con reseñas
- `PUT    /api/propiedades/:id` — actualizar
- `DELETE /api/propiedades/:id` — soft delete (estado='eliminada')


## 2.1 Publicar una propiedad

> *El sistema debe permitir a un anfitrión publicar propiedades con su
> información y servicios disponibles.*

Crea una propiedad nueva en MongoDB y sincroniza el nodo + relación
`(:Usuario)-[:ANFITRIO]->(:Propiedad)` en Neo4j.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
nueva_propiedad = {
    "anfitrion_id":  HOST1_ID,
    "titulo":        "Loft frente al Río - Notebook Test",
    "tipo":          "loft",                       # departamento / casa / loft
    "ubicacion": {
        "ciudad":    "Buenos Aires",
        "pais":      "Argentina",
        "direccion": "Av. Costanera 500",
        "coords": { "type": "Point", "coordinates": [-58.3700, -34.6100] }
    },
    "precio_noche":       72000,
    "cantidad_huespedes": 2,
    "descripcion":        "Loft moderno con vista al río, ideal para parejas.",
    "servicios":          ["wifi", "aire acondicionado", "pileta"],
    "imagen":             "https://images.unsplash.com/photo-1502672260266-1c1ef2d93688?auto=format&fit=crop&w=900&q=80"
}

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.post(f"{API}/propiedades", json=nueva_propiedad))
if r.ok:
    NUEVA_PROP_ID = r.json()["data"]["id"]
    print(f"\n✅ Propiedad creada con ID {NUEVA_PROP_ID}")

## 2.2 Consultar propiedades con filtros combinados

> *El sistema debe permitir consultar propiedades según ubicación, tipo de
> propiedad y precio.*

El backend admite filtros opcionales: `ciudad`, `tipo`, `precioMin`,
`precioMax`, `anfitrion_id`. Si no pasás ninguno, devuelve todas las activas.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
filtros = {
    "ciudad":    "Buenos Aires",      # texto libre, búsqueda case-insensitive
    "tipo":      "departamento",      # opcional
    "precioMax": 100000,              # opcional
    # "precioMin": 30000,             # opcional
    # "anfitrion_id": HOST1_ID,       # opcional
}

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.get(f"{API}/propiedades", params=filtros))
if r.ok:
    print(f"\n📊 {len(r.json()['data'])} propiedades coinciden con los filtros")

## 2.3 Búsqueda geoespacial (radio en km)

> *El sistema debe permitir buscar propiedades por ciudad o mediante
> coordenadas geográficas.*

Usando `lat`, `lng` y `radioKm`, el backend ejecuta un pipeline `$geoNear` en
MongoDB sobre el índice `2dsphere`. Devuelve las propiedades ordenadas por
distancia y agrega el campo `distancia_metros` a cada una.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
geo = {
    "lat":     -34.5890,    # latitud del punto de búsqueda
    "lng":     -58.4210,    # longitud
    "radioKm": 20           # radio en kilómetros
}

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.get(f"{API}/propiedades", params=geo))
if r.ok:
    for p in r.json()["data"]:
        km = p.get("distancia_metros", 0) / 1000
        print(f"  - {p['titulo']}: {km:.2f} km")

## 2.4 Actualizar una propiedad

> *El sistema debe permitir a un anfitrión actualizar o eliminar sus
> propiedades.*

`PUT` admite un patch parcial — cualquier campo se puede modificar (excepto
`id` y `anfitrion_id`).

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
propiedad_id = PROP1_ID   # Depto Palermo
patch = {
    "precio_noche":       75000,
    "cantidad_huespedes": 4,
    "descripcion":        "Descripción actualizada desde el notebook",
    "servicios":          ["wifi", "aire acondicionado", "cocina", "smart TV", "balcón"],
}

# ── Llamada ─────────────────────────────────────────────────────────────
show(requests.put(f"{API}/propiedades/{propiedad_id}", json=patch))

## 2.5 Eliminar una propiedad (soft delete)

`DELETE` no borra físicamente: cambia el `estado` a `"eliminada"`. Las reservas
y reseñas históricas quedan intactas.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
propiedad_id = "PEGAR_ID_AQUI"   # poné el ID de una propiedad que quieras eliminar

# ── Llamada (descomentar para ejecutar) ─────────────────────────────────
# show(requests.delete(f"{API}/propiedades/{propiedad_id}"))
print("⚠️  Esta celda está comentada por seguridad. Descomentá las líneas de arriba para ejecutar.")

## 2.6 Listar todas las propiedades de un anfitrión

> *El sistema debe permitir visualizar todas las propiedades asociadas a un
> anfitrión.*

Filtrando por `anfitrion_id` se obtienen sólo sus publicaciones.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
anfitrion_id = HOST1_ID

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.get(f"{API}/propiedades", params={"anfitrion_id": anfitrion_id}))
if r.ok:
    print(f"\n📊 El anfitrión tiene {len(r.json()['data'])} propiedades activas")

## 2.7 Verificar capacidad máxima de huéspedes

> *El sistema debe permitir definir la capacidad máxima de huéspedes por
> propiedad.*

El campo `cantidad_huespedes` se setea al publicar (sección 2.1) y se respeta
al crear una reserva (sección 3, valida que el `cantidad_huespedes` de la
reserva ≤ capacidad de la propiedad).

In [ ]:
# Demostración: traer todas las propiedades y mostrar su capacidad
r = requests.get(f"{API}/propiedades")
if r.ok:
    print(f"{'Propiedad':<45} {'Ciudad':<20} {'Cap. max'}")
    print("-" * 80)
    for p in r.json()["data"]:
        print(f"{p['titulo'][:43]:<45} {p['ubicacion']['ciudad']:<20} {p['cantidad_huespedes']}")

---

# 3. Gestión de Reservas

Endpoints:
- `POST  /api/reservas` — crear (transacción ACID con pago en Postgres)
- `GET   /api/reservas?huesped_id=...&anfitrion_id=...&propiedad_id=...&estado=...`
- `PATCH /api/reservas/:id/cancelar`
- `PATCH /api/reservas/:id/finalizar`
- `PATCH /api/reservas/:id/pago`


## 3.1 Crear una reserva

> *El sistema debe permitir a un huésped realizar reservas sobre propiedades
> disponibles para un rango de fechas.*

> *El sistema debe permitir indicar la cantidad de huéspedes al momento de
> realizar la reserva.*

> *El sistema debe verificar la disponibilidad de fechas antes de confirmar
> una reserva.*

El backend ejecuta dentro de `BEGIN/COMMIT`:
1. Validar que las fechas no se solapen con otra reserva confirmada.
2. Validar `cantidad_huespedes ≤ propiedad.cantidad_huespedes`.
3. `INSERT` en `reservas` + `INSERT` en `pagos` (uno solo por reserva).
4. Crear relación `(:Usuario)-[:RESERVO]->(:Propiedad)` en Neo4j.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
inicio = date.today() + timedelta(days=7)    # check-in dentro de 7 días
fin    = date.today() + timedelta(days=10)   # check-out dentro de 10 días

reserva = {
    "huesped_id":         GUEST1_ID,
    "propiedad_id":       PROP3_ID,           # Loft Rosario
    "fecha_inicio":       inicio.isoformat(),
    "fecha_fin":          fin.isoformat(),
    "cantidad_huespedes": 2,
    "pago": {
        "metodo": "tarjeta",         # tarjeta / transferencia / efectivo
        "estado": "pendiente"        # pendiente / completado / rechazado
    }
}

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.post(f"{API}/reservas", json=reserva))
if r.ok:
    NUEVA_RESERVA_ID = r.json()["data"]["id"]
    print(f"\n✅ Reserva creada con ID {NUEVA_RESERVA_ID}")

## 3.2 Verificación de disponibilidad — intentar fechas solapadas

> *El sistema debe verificar la disponibilidad de fechas antes de confirmar
> una reserva.*

Si las fechas se solapan con una reserva **confirmada**, el backend responde
HTTP 409 sin crear el registro.

In [ ]:
# Intentamos crear otra reserva sobre la MISMA propiedad y MISMAS fechas
reserva_dup = {**reserva, "huesped_id": GUEST2_ID}   # otro huésped, mismo rango

r = show(requests.post(f"{API}/reservas", json=reserva_dup))
if r.status_code == 409:
    print("\n✅ Validación funcionando: el backend bloqueó la reserva solapada.")

## 3.3 Consultar el estado de una reserva

> *El sistema debe permitir consultar el estado de una reserva.*

`GET /api/reservas` admite filtros por `huesped_id`, `anfitrion_id`,
`propiedad_id`, `estado`. Cada reserva viene hidratada con datos del huésped,
anfitrión y propiedad.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
filtros = {
    "huesped_id": GUEST1_ID,         # opcional
    # "anfitrion_id": HOST1_ID,
    # "propiedad_id": PROP3_ID,
    # "estado": "confirmada",        # confirmada / cancelada / completada
}

# ── Llamada ─────────────────────────────────────────────────────────────
r = show(requests.get(f"{API}/reservas", params=filtros))
if r.ok:
    print(f"\n📊 {len(r.json()['data'])} reservas coinciden")

## 3.4 Cancelar una reserva

> *El sistema debe permitir cancelar una reserva.*

Cambia el estado a `cancelada`. El pago queda en su estado original — no se
modifica automáticamente.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
reserva_id = "PEGAR_ID_RESERVA"   # usá el NUEVA_RESERVA_ID que devolvió 3.1

# ── Llamada ─────────────────────────────────────────────────────────────
# show(requests.patch(f"{API}/reservas/{reserva_id}/cancelar"))
print("⚠️  Comentado por seguridad. Pegá el reserva_id y descomentá la línea.")

## 3.5 Finalizar una reserva (necesario para luego dejar reseña)

`PATCH /api/reservas/:id/finalizar` marca la reserva como `completada` y
actualiza el pago a `completado`. Esto habilita la sección 5 (reseñas).

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
reserva_id = "PEGAR_ID_RESERVA"

# ── Llamada ─────────────────────────────────────────────────────────────
# show(requests.patch(f"{API}/reservas/{reserva_id}/finalizar"))
print("⚠️  Comentado por seguridad. Pegá el reserva_id y descomentá la línea.")

---

# 4. Gestión de Pagos

Los pagos están embebidos en las reservas (relación 1:1, FK con
`ON DELETE CASCADE` en Postgres). No hay endpoint separado de creación: el
pago se crea junto con la reserva (sección 3.1).

Endpoints:
- `GET   /api/reservas` — incluye el sub-objeto `pago` por cada reserva
- `PATCH /api/reservas/:id/pago` — actualizar `estado` y/o `metodo` del pago


## 4.1 Registrar pago al crear reserva

> *El sistema debe registrar el pago asociado a una reserva, incluyendo monto,
> método de pago y estado.*

Ya cubierto en la sección 3.1: cuando creás una reserva, el body incluye
`pago.metodo` y `pago.estado`, y el backend calcula el `monto` como
`noches × precio_noche` dentro de la transacción.

In [ ]:
# Demostración: pedir las reservas y ver el sub-objeto pago de cada una
r = requests.get(f"{API}/reservas")
if r.ok:
    print(f"{'Reserva':<40} {'Monto':>12} {'Método':<14} {'Estado pago'}")
    print("-" * 90)
    for res in r.json()["data"]:
        pago = res["pago"]
        print(f"{res['id'][:36]:<40} ${pago['monto']:>10,.0f}  {pago['metodo']:<14} {pago['estado']}")

## 4.2 Consultar el estado del pago de una reserva

> *El sistema debe permitir consultar el estado del pago de una reserva.*

Filtrando `GET /api/reservas?propiedad_id=...` o por huésped, cada item de la
respuesta incluye el sub-objeto `pago` completo.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
reserva_id = "PEGAR_ID_RESERVA"

# ── Llamada: traemos todas las reservas y filtramos en cliente ──────────
r = requests.get(f"{API}/reservas")
if r.ok:
    encontrada = next((x for x in r.json()["data"] if x["id"] == reserva_id), None)
    if encontrada:
        print(json.dumps(encontrada["pago"], indent=2))
    else:
        print(f"⚠️  No se encontró reserva con ID {reserva_id}")

## 4.3 (Extra) Actualizar el estado de un pago

`PATCH /api/reservas/:id/pago` permite cambiar manualmente el estado o método
del pago.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
reserva_id = "PEGAR_ID_RESERVA"
nuevo_pago = {
    "estado": "completado",       # pendiente / completado / rechazado
    "metodo": "transferencia",    # opcional
}

# ── Llamada ─────────────────────────────────────────────────────────────
# show(requests.patch(f"{API}/reservas/{reserva_id}/pago", json=nuevo_pago))
print("⚠️  Comentado por seguridad. Pegá el reserva_id y descomentá la línea.")

---

# 5. Gestión de Reseñas

Las reseñas viven en **Cassandra** con 3 tablas denormalizadas (una por patrón
de query). Al crearse, se cachea en MongoDB y se recalculan los promedios de
propiedad y anfitrión.

Endpoints:
- `POST /api/resenias` — crear (solo si la reserva está `completada`)
- `GET  /api/resenias` — todas
- `GET  /api/propiedades/:id/resenias` — por propiedad
- `GET  /api/resenias/anfitrion/:id` — por anfitrión


## 5.1 Crear una reseña

> *El sistema debe permitir a un huésped dejar una reseña una vez finalizada
> la reserva.*

> *El sistema debe garantizar que solo un huésped que haya realizado una
> reserva pueda dejar una reseña.*

**Precondiciones** que valida el backend:
1. La reserva existe.
2. El estado de la reserva es `"completada"`.
3. No existe otra reseña sobre la misma reserva (Cassandra
   `resenias_by_reserva`).
4. `calificacion ∈ [1, 5]`.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
nueva_resenia = {
    "reserva_id":   "PEGAR_ID_RESERVA_COMPLETADA",
    "calificacion": 5,                                # 1 a 5
    "comentario":   "Excelente alojamiento, muy recomendable."
}

# ── Llamada ─────────────────────────────────────────────────────────────
# show(requests.post(f"{API}/resenias", json=nueva_resenia))
print("⚠️  Comentado por seguridad. Finalizá una reserva primero (3.5) y pegá su ID acá.")

## 5.2 Consultar las reseñas de una propiedad

> *El sistema debe permitir consultar las reseñas de una propiedad.*

Lee de `resenias_by_propiedad` en Cassandra (partition key = `propiedad_id`).
Devuelve ordenadas por `created_at` descendente.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
propiedad_id = PROP1_ID

# ── Llamada ─────────────────────────────────────────────────────────────
show(requests.get(f"{API}/propiedades/{propiedad_id}/resenias"))

## 5.3 Consultar las reseñas de un anfitrión

Lee de `resenias_by_anfitrion` (otra partition).

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
anfitrion_id = HOST1_ID

# ── Llamada ─────────────────────────────────────────────────────────────
show(requests.get(f"{API}/resenias/anfitrion/{anfitrion_id}"))

## 5.4 Promedio de calificaciones de propiedad y anfitrión

> *El sistema debe calcular y mostrar el promedio de calificaciones de una
> propiedad y de un anfitrión.*

El backend mantiene `promedio_rating` en MongoDB para `propiedades` y
`usuarios`. Se recalcula automáticamente tras cada reseña usando la
colección `resenias_cache`.

In [ ]:
# ── Variables editables ─────────────────────────────────────────────────
propiedad_id = PROP1_ID
anfitrion_id = HOST1_ID

# ── Llamadas ────────────────────────────────────────────────────────────
prop = requests.get(f"{API}/propiedades/{propiedad_id}").json()["data"]
host = requests.get(f"{API}/usuarios/{anfitrion_id}").json()["data"]

print(f"📌 Propiedad: {prop['titulo']}")
print(f"   ⭐ Promedio: {prop['promedio_rating']}")
print()
print(f"📌 Anfitrión: {host['nombre']}")
print(f"   ⭐ Promedio: {host['promedio_rating']}")

## 5.5 Validación: huésped sin reserva no puede reseñar

> *El sistema debe garantizar que solo un huésped que haya realizado una
> reserva pueda dejar una reseña.*

Intentar reseñar una reserva inexistente o no completada debe fallar con
HTTP 404 o 400.

In [ ]:
# Intentamos reseñar una reserva con ID falso → 404
intento_falso = {
    "reserva_id":   "00000000-fake-fake-fake-000000000000",
    "calificacion": 5,
    "comentario":   "esto no debería entrar"
}

r = show(requests.post(f"{API}/resenias", json=intento_falso))
if r.status_code in (400, 404):
    print(f"\n✅ Validación funcionando: el backend rechazó la reseña sin reserva válida ({r.status_code}).")

---

# 🎯 Verificación end-to-end

Después de correr todas las celdas, abrí el frontend en
**http://localhost:5173** y refrescá: deberías ver:

- El nuevo usuario en *Usuarios* (admin).
- La propiedad nueva en *Explorar*.
- La reserva en *Reservas* (logueado como el huésped que la creó).
- El rating actualizado en la tarjeta de la propiedad reseñada.

## Atajos útiles de Docker

```bash
# ver logs del backend en tiempo real (útil mientras corrés celdas)
docker compose logs -f backend

# entrar a mongosh
docker exec -it airbnb-tpo-mongo mongosh airbnb_tpo

# entrar a psql
docker exec -it airbnb-tpo-postgres psql -U airbnb -d airbnb_tpo

# entrar a cqlsh (Cassandra)
docker exec -it airbnb-tpo-cassandra cqlsh -k airbnb_tpo

# entrar a Neo4j browser → http://localhost:7474 (user: neo4j, pass: airbnb_tpo_2026)
```
